# 🖼️ NB-17: Vision-Language VQA – Diagram Question Answering
**Goal:** Render Claude responses as visual 'knowledge cards', then fine-tune a ViT+T5 VQA model.

**Modality:** Vision + Language | **Model:** ViT + T5

In [ ]:
!pip install transformers pillow torch torchvision -q

In [ ]:
import json, re, random
random.seed(42)

def load_dataset(path="claude-opus-4_5-250x_jsonl.txt"):
    """Load Claude thinking dataset from JSONL."""
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def extract_fields(records):
    data = []
    for r in records:
        msgs = r["messages"]
        user = next((m["content"] for m in msgs if m["role"]=="user"), "")
        asst = next((m["content"] for m in msgs if m["role"]=="assistant"), "")
        think = re.findall(r"<think>(.*?)</think>", asst, re.DOTALL)
        response = re.sub(r"<think>.*?</think>", "", asst, flags=re.DOTALL).strip()
        data.append({
            "user": user,
            "assistant_full": asst,
            "thinking": " ".join(think).strip(),
            "response": response
        })
    return data

records = load_dataset()
data = extract_fields(records)
print(f"Loaded {len(data)} examples")
print(f"Sample user: {data[0]['user'][:80]}")
print(f"Sample think: {data[0]['thinking'][:120]}...")
print(f"Sample resp: {data[0]['response'][:120]}...")


In [ ]:
from PIL import Image, ImageDraw, ImageFont
import torch, numpy as np
from transformers import ViTFeatureExtractor, T5Tokenizer, T5ForConditionalGeneration
import torch.nn as nn

def render_card(text, size=(224,224)):
    img = Image.new("RGB", size, (15,15,30))
    d = ImageDraw.Draw(img)
    d.rectangle([5,5,219,219], outline=(0,180,255), width=2)
    lines = [text[i:i+28] for i in range(0, min(len(text),280), 28)]
    for j, line in enumerate(lines[:8]):
        d.text((12, 20 + j*24), line, fill=(180,220,255))
    return img

images = [render_card(d["response"][:280]) for d in data]
questions = [d["user"] for d in data]
answers = [d["response"][:200] for d in data]

# ViT encoder
vit_extractor = ViTFeatureExtractor.from_pretrained("google/vit-base-patch16-224")
pixel_values = torch.tensor(np.stack([vit_extractor(img)["pixel_values"][0] for img in images]))

# T5 decoder
t5_tok = T5Tokenizer.from_pretrained("google/flan-t5-small")
t5_model = T5ForConditionalGeneration.from_pretrained("google/flan-t5-small")
from transformers import ViTModel
vit = ViTModel.from_pretrained("google/vit-base-patch16-224")

class VQAModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = vit
        self.t5  = t5_model
        self.proj = nn.Linear(768, 512)  # ViT-768 → T5-512
    def forward(self, pixel_values, input_ids, attention_mask, labels=None):
        vis = self.proj(self.vit(pixel_values=pixel_values).last_hidden_state[:,0,:])
        enc = self.t5.encoder(input_ids=input_ids, attention_mask=attention_mask)
        # Fuse visual + text
        fused = torch.cat([vis.unsqueeze(1), enc.last_hidden_state], dim=1)
        from transformers.modeling_outputs import BaseModelOutput
        enc_out = BaseModelOutput(last_hidden_state=fused)
        ext_mask = torch.ones(fused.shape[:2], device=fused.device)
        return self.t5(encoder_outputs=enc_out, encoder_attention_mask=ext_mask, labels=labels)

model = VQAModel()
print("✅ VQA model built — ViT encoder + T5 decoder")
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")


In [ ]:
from torch.utils.data import Dataset as TDS, DataLoader

enc_q = t5_tok(questions, truncation=True, padding="max_length", max_length=64, return_tensors="pt")
enc_a = t5_tok(answers, truncation=True, padding="max_length", max_length=128, return_tensors="pt")

class VQADS(TDS):
    def __len__(self): return len(questions)
    def __getitem__(self, i):
        return pixel_values[i], enc_q["input_ids"][i], enc_q["attention_mask"][i], enc_a["input_ids"][i]

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-5)

for epoch in range(3):
    loader = DataLoader(VQADS(), batch_size=4, shuffle=True)
    total = 0
    for pv, iids, amask, labs in loader:
        labs[labs == t5_tok.pad_token_id] = -100
        out = model(pv.to(device), iids.to(device), amask.to(device), labs.to(device))
        opt.zero_grad(); out.loss.backward(); opt.step()
        total += out.loss.item()
    print(f"Epoch {epoch+1} VQA Loss: {total/len(loader):.4f}")
print("✅ VQA model trained!")
